# [P1] Q1 A + C: neuron models and validation

**Owner:** Part 1
**Temporary part notebook.** At integration it is merged into `Q1_neural_spikes.ipynb` and this file is deleted.

The model functions live in `q1_models.py`, because the other parts import them from there
(the agreed contract: `make_rngs(seed, n)` and `simulate_*(T, rng, rate_hz=...)`).
In this notebook every function's code is printed next to its explanation, so the notebook can be
read on its own. At integration the code is pasted into the final notebook in the same places.

**Contents**
* **Q1 A – the models:** sections A1–A12
* **Q1 C – testing our assumptions:** sections C1–C6

## [P1] Q1 A1. Imports
`q1_models.py` holds the model functions and their fixed parameters; `inspect` is used to print each function's code next to its explanation.

In [ ]:
import inspect
import numpy as np
import matplotlib.pyplot as plt

# the model functions and their fixed parameters live in q1_models.py (team contract)
from q1_models import (DEAD_TIME, BURN_IN, K_REGULAR, K_IRREGULAR,
                       MIN_SPIKES, EXTRA_SPIKES_MEAN, MEAN_SPIKES_PER_BURST,
                       INTRA_MEAN, INTRA_K, QUIET_MIN, QUIET_K,
                       make_rngs, draw_isis, isis_to_spike_times, check_spike_times,
                       simulate_regular, simulate_irregular,
                       draw_shifted_gamma, bursting_timing, simulate_bursting)

## [P1] Q1 A2. Parameters
**Decision:** all three neurons fire at the same mean rate, 10 Hz.
**Why:** if the rate is equal, any difference between the neurons must come from the
*timing* of the spikes.

In [ ]:
# length of the recording, in seconds
DURATION = 5.0

# mean firing rate, in spikes per second (Hz)
RATE = 10.0

# mean interval between spikes = 0.1 s = 100 ms
MEAN_ISI = 1 / RATE


# master random seed for the whole notebook
SEED = 2026

# fixed model parameters, defined in q1_models.py:
# DEAD_TIME = refractory period, BURN_IN = warm-up before the recording
print(f"DEAD_TIME = {DEAD_TIME * 1000:.0f} ms, BURN_IN = {BURN_IN:.0f} s")

## [P1] Q1 A3. Random generators (seeds)
**Decision:** one master seed, split into an independent random generator for each neuron.
**Why:** the same seed always gives the same spike trains (reproducibility), and changing
one neuron does not change the random numbers that the other neurons receive.

In [ ]:
# one master seed -> 3 independent random generators (make_rngs uses SeedSequence.spawn)
print(inspect.getsource(make_rngs))
rng_regular, rng_irregular, rng_bursting = make_rngs(SEED, 3)

## [P1] Q1 A4. Building block 1 – drawing ISIs
**Decision:** each ISI = `DEAD_TIME` + a random number from a **Gamma distribution**.

* **Shape `k`** controls how variable the intervals are:
  large `k` → nearly equal intervals (regular); `k = 1` → very variable (irregular, Poisson-like).
* **Scale `theta`** is chosen so that the mean ISI is always exactly 100 ms, whatever `k` is.
* Adding `DEAD_TIME` guarantees that no interval is shorter than 2 ms.

**Why Gamma:** one distribution covers both the regular and the irregular neuron;
only the parameter `k` changes between them.

In [ ]:
# draw_isis is defined in q1_models.py (team contract); its code is shown here:
print(inspect.getsource(draw_isis))

## [P1] Q1 A5. Building block 2 – from ISIs to spike times
spike times are the **cumulative sum** of the ISIs, starting from a random first spike.

first spike at 38 ms, ISIs = 100, 98, 103 ms → spike times = 38, 138, 236, 339 ms.

* Every ISI is positive, so the times come out **already in order** – no sorting needed.
* **Burn-in:** a real recording starts while the neuron is already active. So the neuron starts
  firing `BURN_IN` = 2 s *before* the recording (first spike at a random time), and every spike
  before t = 0 is dropped.

**Why burn-in and not simply "first spike uniform in [0, 100 ms)"?** Our first version did that.
For the regular neuron it is correct, but for the irregular neuron it puts the first spike too early
on average (50 ms instead of ~100 ms), which adds about half a spike to every recording.
Check 5b in Q1 C detected this (10.1 Hz instead of 10 Hz at 5 s, 11 Hz at 0.5 s).
After 2 s (~20 spikes) the neuron has "forgotten" how it started, so the bias disappears.

In [ ]:
# isis_to_spike_times is defined in q1_models.py (team contract); its code is shown here:
print(inspect.getsource(isis_to_spike_times))

## [P1] Q1 A6. Building block 3 – checking that spike times are valid
after every simulation we check the basic rules with `assert`.
If a rule is broken, the code stops immediately with a message, instead of silently
producing wrong data.

In [ ]:
# check_spike_times is defined in q1_models.py (team contract); its code is shown here:
print(inspect.getsource(check_spike_times))

## [P1] Q1 A7. The regular neuron
Gamma shape `k = 400`.
with `k = 400` the coefficient of variation of the ISIs is
CV = (MEAN_ISI − DEAD_TIME) / (MEAN_ISI · √k) ≈ 0.049, i.e. intervals vary by only about 5%
around 100 ms – "nearly constant intervals".

**Deterministic:** rate, duration, dead time, `k`. **Stochastic:** the first spike time and every ISI.

In [ ]:
# simulate_regular is defined in q1_models.py (team contract); its code is shown here:
print(inspect.getsource(simulate_regular))

# K_REGULAR = 400 (defined in q1_models.py)
spikes_regular = simulate_regular(DURATION, rng_regular, rate_hz=RATE)

## [P1] Q1 A8. Quick look at the regular neuron
A first sanity check (the full validation is in Q1 C): the numbers should be close to the
values we designed the model for.

In [ ]:
isis = np.diff(spikes_regular)                                       # intervals between spikes
cv_measured = isis.std(ddof=1) / isis.mean()                         # CV = std / mean
cv_theory = (MEAN_ISI - DEAD_TIME) / (MEAN_ISI * np.sqrt(K_REGULAR))  # CV we designed for

print(f"Number of spikes: {len(spikes_regular)}   (expected about {RATE * DURATION:.0f})")
print(f"Mean ISI:         {isis.mean() * 1000:.1f} ms (expected 100 ms)")
print(f"CV of ISI:        {cv_measured:.3f}    (expected {cv_theory:.3f})")

In [ ]:
fig, (ax_raster, ax_hist) = plt.subplots(1, 2, figsize=(11, 2.8),
                                         gridspec_kw={"width_ratios": [2.5, 1]})

# Left: raster – one vertical line per spike
ax_raster.eventplot(spikes_regular, colors="black")
ax_raster.set(xlim=(0, DURATION), yticks=[], xlabel="Time (s)", title="Regular neuron – spike train")

# Right: histogram of the intervals between spikes
ax_hist.hist(isis * 1000, bins=15, color="gray", edgecolor="black")
ax_hist.set(xlabel="ISI (ms)", ylabel="Count", title="ISI histogram")

fig.tight_layout()
plt.show()

## [P1] Q1 A9. The irregular neuron

With `k = 1` the Gamma distribution is the **exponential distribution**, so (apart from the 2 ms
dead time) this is a **Poisson process**: the chance of a spike in the next moment does not
depend on how long ago the last spike was ("no memory").

the Poisson process is the standard model of irregular, stochastic firing.
Its expected CV is 1. Because of the 2 ms dead time it is slightly lower:
CV = (100 − 2) / (100 · √1) = 0.98.

**What changes compared with the regular neuron:** only `k` (400 → 1).
Rate, duration, dead time and the way spike times are built stay the same.

In [ ]:
# simulate_irregular is defined in q1_models.py (team contract); its code is shown here:
print(inspect.getsource(simulate_irregular))

# K_IRREGULAR = 1 (defined in q1_models.py)
spikes_irregular = simulate_irregular(DURATION, rng_irregular, rate_hz=RATE)

### Quick look at the irregular neuron

In [ ]:
# intervals between spikes
isis_irregular = np.diff(spikes_irregular)
# CV = std / mean
cv_irregular = isis_irregular.std(ddof=1) / isis_irregular.mean()
# CV we designed for
cv_irregular_theory = (MEAN_ISI - DEAD_TIME) / (MEAN_ISI * np.sqrt(K_IRREGULAR))

print(f"Number of spikes: {len(spikes_irregular)}   (expected about {RATE * DURATION:.0f})")
print(f"Mean ISI:         {isis_irregular.mean() * 1000:.1f} ms (expected 100 ms)")
print(f"ISI range:        {isis_irregular.min() * 1000:.1f} – {isis_irregular.max() * 1000:.1f} ms")
print(f"CV of ISI:        {cv_irregular:.3f}    (expected {cv_irregular_theory:.3f})")

In [ ]:
fig, (ax_raster, ax_hist) = plt.subplots(1, 2, figsize=(11, 2.8),
                                         gridspec_kw={"width_ratios": [2.5, 1]})

# Left: raster – one vertical line per spike
ax_raster.eventplot(spikes_irregular, colors="black")
ax_raster.set(xlim=(0, DURATION), yticks=[], xlabel="Time (s)", title="Irregular neuron – spike train")

# Right: histogram of the intervals between spikes
ax_hist.hist(isis_irregular * 1000, bins=15, color="gray", edgecolor="black")
ax_hist.set(xlabel="ISI (ms)", ylabel="Count", title="ISI histogram")

fig.tight_layout()
plt.show()

the spikes are not evenly spaced – some are very close together, with
long gaps elsewhere. The ISI histogram is highest at short intervals and decays (exponential shape),
unlike the narrow peak around 100 ms of the regular neuron.
With only ~50 spikes in 5 s, the spike count and CV of a single recording can differ noticeably
from the expected values; Q1 C checks them over many realizations.

## [P1] Q1 A10. The bursting neuron
A bursting neuron fires **short groups of fast spikes (bursts)** separated by **quiet periods**.
So it has two very different kinds of intervals, and we model each one separately:

| Part of the model | Decision | Why |
|---|---|---|
| Spikes per burst | n = 2 + Poisson(2) → between 2 and ~8, mean 4 | a burst has at least 2 spikes; the size varies from burst to burst |
| ISI **inside** a burst | mean 5 ms (≈200 Hz), CV = 0.2, minimum 2 ms | typical intra-burst frequency of bursting cells is 100–300 Hz |
| Quiet period **between** bursts | at least 50 ms + Gamma(k = 4) (CV 0.5 of the random part) | bursts never merge, and they come at roughly (not exactly) regular times |
| Start | bursts start `BURN_IN` = 2 s before the recording, first burst at a random time | same burn-in as the other neurons |

**Keeping the same mean rate (10 Hz):** 4 spikes per burst on average → one burst every 400 ms on average.
A burst lasts on average 3 × 5 = 15 ms, so the mean quiet period is 400 − 15 = 385 ms.

**Deterministic:** all the numbers in the table. **Stochastic:** the number of spikes in each burst,
every ISI inside a burst, every quiet period, and the start time.

In [ ]:
# the bursting parameters are defined in q1_models.py:
print(f"spikes per burst = {MIN_SPIKES} + Poisson({EXTRA_SPIKES_MEAN})  -> mean {MEAN_SPIKES_PER_BURST}")
print(f"ISI inside a burst: mean {INTRA_MEAN * 1000:.0f} ms, minimum {DEAD_TIME * 1000:.0f} ms, Gamma shape {INTRA_K}")
print(f"quiet period: at least {QUIET_MIN * 1000:.0f} ms, Gamma shape {QUIET_K}")

# the burst cycle and quiet period that keep the mean rate at RATE
print(inspect.getsource(bursting_timing))
MEAN_CYCLE, QUIET_MEAN = bursting_timing(RATE)
print(f"mean cycle = {MEAN_CYCLE * 1000:.0f} ms, mean quiet period = {QUIET_MEAN * 1000:.0f} ms")

### A general shifted-Gamma helper
Building block 1 (`draw_isis`) always uses the global mean ISI (100 ms) and the 2 ms dead time.
The bursting neuron needs the same idea with **other** values (5 ms inside bursts, 385 ms quiet
periods), so we write a slightly more general version: `minimum + Gamma`, with a chosen mean.

In [ ]:
# draw_shifted_gamma is defined in q1_models.py (team contract); its code is shown here:
print(inspect.getsource(draw_shifted_gamma))

### Building the bursts one after another
1. Start the first burst at a random time, `BURN_IN` seconds before the recording.
2. Draw how many spikes this burst has, and the intervals between them → the spikes of the burst.
3. Draw a quiet period; the next burst starts that long after the **last spike** of this burst.
4. Repeat until we pass the end of the recording, then keep only spikes in `[0, DURATION)`.

Because every interval (inside and between bursts) is positive, the spike times are in order.
A burst that is in progress at t = 0 or at t = 5 s is cut – exactly like in a real recording.

In [ ]:
# simulate_bursting is defined in q1_models.py (team contract); its code is shown here:
print(inspect.getsource(simulate_bursting))

spikes_bursting = simulate_bursting(DURATION, rng_bursting, rate_hz=RATE)

### Quick look at the bursting neuron
Two different scales of ISIs (~5 ms and ~400 ms) do not fit well on a normal axis, so the
histogram uses a **logarithmic time axis**. On it we expect **two separate peaks**:
the left one = intervals inside bursts, the right one = intervals between bursts.

In [ ]:
# intervals between spikes
isis_bursting = np.diff(spikes_bursting)
# CV = std / mean
cv_bursting = isis_bursting.std(ddof=1) / isis_bursting.mean()
# intervals shorter than the minimum quiet period must be inside a burst
n_within = np.sum(isis_bursting < QUIET_MIN)

print(f"Number of spikes:      {len(spikes_bursting)}   (expected about {RATE * DURATION:.0f})")
print(f"Mean ISI:              {isis_bursting.mean() * 1000:.1f} ms (expected 100 ms)")
print(f"ISI range:             {isis_bursting.min() * 1000:.1f} – {isis_bursting.max() * 1000:.1f} ms")
print(f"CV of ISI:             {cv_bursting:.3f}   (above 1: more variable than Poisson)")
print(f"ISIs inside bursts:    {n_within}   (< {QUIET_MIN * 1000:.0f} ms)")
print(f"ISIs between bursts:   {len(isis_bursting) - n_within}")

In [ ]:
fig, (ax_raster, ax_hist) = plt.subplots(1, 2, figsize=(11, 2.8),
                                         gridspec_kw={"width_ratios": [2.5, 1]})

# Left: raster – one vertical line per spike
ax_raster.eventplot(spikes_bursting, colors="black")
ax_raster.set(xlim=(0, DURATION), yticks=[], xlabel="Time (s)", title="Bursting neuron – spike train")

# Right: ISI histogram on a log axis (bins equally spaced on the log scale, 1 ms to 1 s)
log_bins = np.logspace(0, 3, 30)
ax_hist.hist(isis_bursting * 1000, bins=log_bins, color="gray", edgecolor="black")
ax_hist.set(xscale="log", xlabel="ISI (ms, log scale)", ylabel="Count", title="ISI histogram")

fig.tight_layout()
plt.show()

**What to look for:** in the raster, spikes appear in tight clusters with silence between them.
The ISI histogram is **bimodal** – a peak at a few ms (inside bursts) and a peak at hundreds of ms
(between bursts), with an empty gap between them (no interval between bursts is shorter than 50 ms). The mean rate is still ~10 Hz,
like the other two neurons, but the CV is above 1.

## [P1] Q1 A11. Summary of the models and parameters
All times are in ms unless stated otherwise. All three neurons: recording of 5 s, mean rate 10 Hz,
minimum interval between any two spikes 2 ms, 2 s burn-in before the recording, master seed 2026.

| Neuron | Parameter | Value | Deterministic / Stochastic |
|---|---|---|---|
| All | Recording duration | 5 s | Deterministic |
| All | Mean firing rate | 10 Hz (mean ISI 100 ms) | Deterministic |
| All | Dead time (minimum ISI) | 2 ms | Deterministic |
| All | Burn-in before the recording | 2 s | Deterministic |
| All | Time of the first spike / burst (before the burn-in) | Uniform in [0, mean cycle) | **Stochastic** |
| Regular | ISI | 2 + Gamma(k = 400), mean 100 → CV ≈ 0.049 | **Stochastic** (each ISI) |
| Regular | Gamma shape k | 400 | Deterministic |
| Irregular | ISI | 2 + Gamma(k = 1) = 2 + Exponential, mean 100 → CV ≈ 0.98 | **Stochastic** (each ISI) |
| Irregular | Gamma shape k | 1 | Deterministic |
| Bursting | Spikes per burst | 2 + Poisson(2), mean 4 | **Stochastic** (each burst) |
| Bursting | ISI inside a burst | 2 + Gamma(k = 9), mean 5 → CV = 0.2 | **Stochastic** (each ISI) |
| Bursting | Quiet period between bursts | 50 + Gamma(k = 4), mean 385 | **Stochastic** (each gap) |
| Bursting | Mean burst cycle | 400 ms (= 4 spikes / 10 Hz) | Deterministic |

**How spike times are kept valid and in order:** every interval is positive (it always includes the
2 ms dead time), spike times are a cumulative sum of intervals, spikes before 0 s and after 5 s are removed, and
`check_spike_times` stops the code if any rule is broken.

## [P1] Q1 A12. The three neurons side by side
**Decision:** each neuron has a fixed colour that is used in all figures, and all rasters share
the same time axis, so the neurons can be compared directly. The figure is saved as PNG (200 dpi)
in the `figures/` folder.

The left panel shows the whole 5 s recording. Spikes inside a burst are only ~5 ms apart, so at
this scale each burst looks like one thick line; the right panel zooms in on 100 ms around the
first burst, where its individual spikes become visible.

In [ ]:
from pathlib import Path

# fixed colour for each neuron (used in every figure)
NEURON_COLORS = {"Regular": "tab:blue", "Irregular": "tab:orange", "Bursting": "tab:green"}

# folder for saved figures (works from parts/ now and from the project root after merging)
FIG_DIR = Path("figures") if Path("figures").is_dir() else Path("../figures")

In [ ]:
all_spikes = {"Regular": spikes_regular,
              "Irregular": spikes_irregular,
              "Bursting": spikes_bursting}

fig, (ax_full, ax_zoom) = plt.subplots(1, 2, figsize=(12, 3.2),
                                       gridspec_kw={"width_ratios": [3, 1]}, sharey=True)

# zoom window: 100 ms that starts 20 ms before the first burst
zoom_start = spikes_bursting[0] - 0.020
zoom_window = (zoom_start, zoom_start + 0.100)

for ax, x_limits, title in [(ax_full, (0, DURATION), "Whole recording (5 s)"),
                            (ax_zoom, zoom_window, "Zoom: 100 ms around the first burst")]:
    # one row per neuron, each in its own colour
    ax.eventplot(list(all_spikes.values()), colors=list(NEURON_COLORS.values()),
                 lineoffsets=[2, 1, 0], linelengths=0.7)
    ax.set(xlim=x_limits, xlabel="Time (s)", title=title)

# neuron names on the y axis, top to bottom
ax_full.set(yticks=[2, 1, 0], yticklabels=list(all_spikes.keys()))

fig.tight_layout()
fig.savefig(FIG_DIR / "q1_raster_three_neurons.png", dpi=200)
plt.show()

In [ ]:
# spike counts: the mean rate is similar, the timing is not
for name, spikes in all_spikes.items():
    print(f"{name:10s} {len(spikes):3d} spikes  ->  {len(spikes) / DURATION:4.1f} Hz")

**What to look for:** all three neurons fire a similar number of spikes in 5 s (about 10 Hz),
yet they look completely different: evenly spaced (regular), randomly scattered (irregular),
and grouped in tight clusters with silence between them (bursting).
So the mean firing rate alone cannot tell them apart – the difference is in the **timing** of the
spikes, which Q1 B analyses in detail.

## [P1] Q1 C – Testing our assumptions (Part 1)

A simulation can produce plausible-looking data while implementing the wrong model.
Each check below asks a question that **could have a wrong answer** if the model had a bug,
and has an **acceptance criterion decided before running it**.

**The three main checks** (one of each type required by the assignment):

| Type | Main check |
|---|---|
| Temporal | The regular neuron is really regular |
| Statistical | ISI variability follows the theory when the Gamma shape `k` changes |
| Edge case | Changing the recording duration (from 0.01 s to 500 s) |

**Contents**
1. Setup – number of realizations and the report helper
2. Temporal checks
   * 2.1 Order, positivity, recording window and minimum ISI
   * 2.2 **(main)** The regular neuron is really regular
   * 2.3 Burst structure: bursts do not merge
3. Reproducibility – same seed gives the same result
4. Statistical checks
   * 4.1 **(main)** Variability scan: CV follows the theory for k = 0.5 … 400
   * 4.2 Shape of the irregular ISI distribution (Kolmogorov–Smirnov test)
5. **(main)** Edge case: changing the recording duration
6. Summary of all checks

## [P1] Q1 C1. Setup
The checks use the model functions of Q1 A above (imported from `q1_models.py`), so they test
*exactly* the code that produced our results.

**Number of realizations:** 1000 per neuron. A simulation takes milliseconds, so there is no
reason to use fewer; more realizations make each check more sensitive to real bugs.

In [ ]:
# number of independent realizations for every check
N_REALIZATIONS = 1000

# seed of the checks (different from the seed of Q1 A)
CHECK_SEED = 12345

# the three simulation functions, by name
SIMULATORS = {"Regular": simulate_regular,
              "Irregular": simulate_irregular,
              "Bursting": simulate_bursting}

In [ ]:
def many_realizations(simulate, n=N_REALIZATIONS, seed=CHECK_SEED):
    """Run one simulator n times with one random generator; return a list of spike-time arrays."""
    rng = np.random.default_rng(seed)
    return [simulate(DURATION, rng, rate_hz=RATE) for _ in range(n)]

# 1000 realizations of each neuron, used by several checks below
realizations = {name: many_realizations(sim) for name, sim in SIMULATORS.items()}

In [ ]:
# list that collects the result of every check, for a summary at the end
results = []

def report(check, passed, details):
    """Print the result of one check and remember it."""
    status = "PASS" if passed else "FAIL"
    results.append({"check": check, "result": status, "details": details})
    print(f"[{status}] {check}: {details}")

## [P1] Q1 C2. Temporal checks

### C2.1 Order, positivity, recording window and minimum ISI
**Question:** in every realization of every neuron, are all spike times inside [0, 5 s), strictly
increasing, and never closer than the 2 ms dead time?

a negative or zero ISI would mean two spikes at the same moment or time going
backwards; spikes outside [0, 5) would mean the recording window is wrong.

**Acceptance criterion:** zero violations in all 3 × 1000 realizations.

In [ ]:
for name, trains in realizations.items():
    # join all realizations of this neuron into one array of times and one array of ISIs
    all_times = np.concatenate(trains)
    all_isis = np.concatenate([np.diff(t) for t in trains])

    # count violations of each rule
    n_outside = np.sum((all_times < 0) | (all_times >= DURATION))
    n_not_increasing = np.sum(all_isis <= 0)
    n_below_dead_time = np.sum(all_isis < DEAD_TIME - 1e-12)

    n_violations = n_outside + n_not_increasing + n_below_dead_time
    report(f"2.1 valid spike times ({name})",
           n_violations == 0,
           f"{len(all_isis)} ISIs checked, shortest = {all_isis.min() * 1000:.2f} ms, "
           f"violations = {n_violations}")

### C2.2 (main temporal check) The regular neuron is really regular
**Question:** are the intervals of the regular neuron really close to 100 ms?

"regular" was the whole point of this neuron. If, for example, the shape `k`
were passed in the wrong place, the code would still run but the intervals would be spread out.

**Acceptance criterion** (theory: CV ≈ 0.049, so ±20 % is about 4 standard deviations):
* the median ISI is between 98 and 102 ms, and
* fewer than 0.1 % of ISIs are outside 80–120 ms.

apply the same test to the irregular neuron, where it **must say "not regular"**.
This shows that the test can actually detect a neuron that is not regular (a test that always
passes would be useless).

In [ ]:
def is_regular(trains):
    """Apply the regularity criterion; return (passed, median ISI in ms, fraction outside 80-120 ms)."""
    isis_ms = np.concatenate([np.diff(t) for t in trains]) * 1000
    median_isi = np.median(isis_ms)
    fraction_outside = np.mean((isis_ms < 80) | (isis_ms > 120))
    passed = (98 <= median_isi <= 102) and (fraction_outside < 0.001)
    return passed, median_isi, fraction_outside

# the regular neuron must pass
passed, median_isi, fraction_outside = is_regular(realizations["Regular"])
report("2.2 regularity (Regular)", passed,
       f"median ISI = {median_isi:.1f} ms, outside 80-120 ms = {fraction_outside * 100:.3f} %")

# control: the irregular neuron must NOT pass
passed, median_isi, fraction_outside = is_regular(realizations["Irregular"])
report("2.2 control: test rejects the Irregular neuron", not passed,
       f"median ISI = {median_isi:.1f} ms, outside 80-120 ms = {fraction_outside * 100:.1f} %")

### C2.3 Burst structure: bursts do not merge
**Question:** are bursts always clearly separated by a quiet period?

In the model, intervals inside a burst are about 5 ms (practically never above ~20 ms), and every
quiet period is at least 50 ms. So **no ISI should fall between 20 and 50 ms**. If, for example,
the quiet period were measured from the *start* of a burst instead of its *last spike*, long bursts
would overlap the next one and ISIs in this "forbidden zone" would appear.

We also split each train into bursts (a new burst starts after any ISI ≥ 50 ms) and check that every
burst has at least 2 spikes. The first and the last bursts are excluded, because they may be cut by the
start or the end of the recording.

**Acceptance criterion:** zero ISIs in [20, 50) ms, and zero complete bursts with fewer than 2 spikes.

**Is the check sensitive?** We tested it once on a deliberately broken model (the 50 ms minimum
quiet period removed): 1000 realizations then gave 21 ISIs in the 20–50 ms zone, so the check fails
as it should.

In [ ]:
def split_into_bursts(spike_times, gap=QUIET_MIN):
    """Split spike times into bursts: a new burst starts after an ISI >= gap."""
    # positions where a new burst starts
    new_burst = np.where(np.diff(spike_times) >= gap)[0] + 1
    return np.split(spike_times, new_burst)

n_forbidden = 0
n_small_bursts = 0
n_bursts = 0
for train in realizations["Bursting"]:
    isis = np.diff(train)
    # ISIs in the forbidden zone between 20 and 50 ms
    n_forbidden += np.sum((isis >= 0.020) & (isis < QUIET_MIN))

    # all bursts except the first and last (which may be cut at 0 s or 5 s)
    complete_bursts = split_into_bursts(train)[1:-1]
    n_bursts += len(complete_bursts)
    n_small_bursts += sum(len(b) < MIN_SPIKES for b in complete_bursts)

report("2.3 burst structure (Bursting)",
       n_forbidden == 0 and n_small_bursts == 0,
       f"{n_bursts} complete bursts, ISIs in 20-50 ms = {n_forbidden}, "
       f"bursts with < 2 spikes = {n_small_bursts}")

## [P1] Q1 C3. Reproducibility – same seed gives the same result
**Question:** does the same seed always give exactly the same spike trains, and does a different
seed give different ones?

We also check the benefit of giving each neuron its own random stream (`SeedSequence.spawn`):
changing a parameter of the **regular** neuron must not change the **bursting** neuron.

`np.array_equal` is True for the same seed and False for a different seed,
for all three neurons; and the bursting neuron is identical when only the regular neuron's `k` changes.

In [ ]:
def simulate_all(seed, k_regular=K_REGULAR):
    """Simulate the three neurons from one master seed, exactly as in Q1 A."""
    rngs = make_rngs(seed, 3)
    return {"Regular": simulate_regular(DURATION, rngs[0], rate_hz=RATE, k=k_regular),
            "Irregular": simulate_irregular(DURATION, rngs[1], rate_hz=RATE),
            "Bursting": simulate_bursting(DURATION, rngs[2], rate_hz=RATE)}

run_a = simulate_all(SEED)
# same seed
run_b = simulate_all(SEED)
# different seed
run_c = simulate_all(SEED + 1)

for name in SIMULATORS:
    same = np.array_equal(run_a[name], run_b[name])
    different = not np.array_equal(run_a[name], run_c[name])
    report(f"3 reproducibility ({name})", same and different,
           f"same seed -> identical: {same}, different seed -> different: {different}")

# change only the regular neuron (k = 4 instead of 400)
run_d = simulate_all(SEED, k_regular=4)
unchanged = np.array_equal(run_a["Bursting"], run_d["Bursting"])
report("3 independent random streams", unchanged,
       f"bursting neuron unchanged when regular k changes 400 -> 4: {unchanged}")

## [P1] Q1 C4. Statistical checks

### C4.1 (main statistical check) Variability scan: does the CV follow the theory?
**Question:** when we change the Gamma shape `k`, does the variability of the ISIs change exactly
as the model predicts? (Assignment: *"Does increasing variability affect the ISI distribution as expected?"*)

he CV measured from one 5 s recording (~50 spikes) is
itself random, and its spread is very different for k = 0.5 and k = 400. So we compare the theory with
the actual spread of the estimates.

**Acceptance criterion:** for every `k`, the theoretical CV lies inside the central 95 % of the
1000 CV estimates (between the 2.5th and 97.5th percentiles).

In [ ]:
# Gamma shapes to scan: very irregular ... very regular
K_VALUES = [0.5, 1, 4, 16, 400]

def cv_of_train(spike_times):
    """CV of the ISIs of one spike train."""
    isis = np.diff(spike_times)
    return isis.std(ddof=1) / isis.mean()

# for every k: 1000 realizations, one CV per realization
cv_estimates = {}
isis_by_k = {}
for k in K_VALUES:
    rng = np.random.default_rng(CHECK_SEED)
    trains = [isis_to_spike_times(k, DURATION, rng, RATE) for _ in range(N_REALIZATIONS)]
    cv_estimates[k] = np.array([cv_of_train(t) for t in trains])
    isis_by_k[k] = np.concatenate([np.diff(t) for t in trains])

In [ ]:
for k in K_VALUES:
    cv_theory = (MEAN_ISI - DEAD_TIME) / (MEAN_ISI * np.sqrt(k))
    # central 95 % of the 1000 estimates
    low, high = np.percentile(cv_estimates[k], [2.5, 97.5])
    report(f"4.1 CV scan (k = {k})", low <= cv_theory <= high,
           f"theory CV = {cv_theory:.3f}, median estimate = {np.median(cv_estimates[k]):.3f}, "
           f"95 % range = [{low:.3f}, {high:.3f}]")

In [ ]:
fig, (ax_cv, ax_hist) = plt.subplots(1, 2, figsize=(12, 3.8))

# Left: CV against k – theory line and the spread of the estimates
k_line = np.logspace(np.log10(0.4), np.log10(500), 100)
ax_cv.plot(k_line, (MEAN_ISI - DEAD_TIME) / (MEAN_ISI * np.sqrt(k_line)), color="black", label="theory")
for i, k in enumerate(K_VALUES):
    low, med, high = np.percentile(cv_estimates[k], [2.5, 50, 97.5])
    ax_cv.errorbar(k, med, yerr=[[med - low], [high - med]], fmt="o", color="tab:red", capsize=4,
                   label="estimates (median, 95 %)" if i == 0 else None)
ax_cv.set(xscale="log", yscale="log", xlabel="Gamma shape k", ylabel="CV of ISI",
          title="CV falls like 1/√k")
ax_cv.legend()

# Right: the ISI distribution narrows as k grows
bins = np.linspace(0, 400, 81)
for k in [1, 4, 16, 400]:
    ax_hist.hist(isis_by_k[k] * 1000, bins=bins, density=True, histtype="step", linewidth=1.5,
                 label=f"k = {k}")
ax_hist.set(xlabel="ISI (ms)", ylabel="Density", title="ISI distribution narrows as k grows")
ax_hist.legend()

fig.tight_layout()
fig.savefig(FIG_DIR / "q1_check_cv_scan.png", dpi=200)
plt.show()

### C4.2 Shape of the irregular ISI distribution (Kolmogorov–Smirnov test)
**Question:** are the ISIs of the irregular neuron really distributed as 2 ms + Exponential(mean 98 ms)?

The KS test compares the **whole shape** of the ISI
distribution with the theoretical one.

we run a KS test on each of the 1000 realizations. If the model is right, each p-value is
equally likely to be anywhere between 0 and 1, so:
* about 5 % of the realizations should have p < 0.05 (by chance), and
* the 1000 p-values should be uniformly distributed.

**Acceptance criterion:** the fraction with p < 0.05 is inside the range expected by chance for
1000 tests (binomial 95 % range), and a KS test of the p-values against the uniform distribution
gives p > 0.01.

In [ ]:
from scipy import stats

# theoretical ISI distribution of the irregular neuron: 2 ms + Exponential(mean 98 ms)
theory_isi = stats.expon(loc=DEAD_TIME, scale=MEAN_ISI - DEAD_TIME)

# one KS test per realization
p_values = np.array([stats.kstest(np.diff(t), theory_isi.cdf).pvalue
                     for t in realizations["Irregular"]])

# range of "fraction below 0.05" that is expected by chance in 1000 tests
low, high = stats.binom.ppf([0.025, 0.975], N_REALIZATIONS, 0.05) / N_REALIZATIONS
fraction_small = np.mean(p_values < 0.05)
uniformity_p = stats.kstest(p_values, "uniform").pvalue

report("4.2 KS shape (Irregular)",
       (low <= fraction_small <= high) and uniformity_p > 0.01,
       f"p < 0.05 in {fraction_small * 100:.1f} % of realizations (expected {low * 100:.1f}-{high * 100:.1f} %), "
       f"uniformity of p-values: p = {uniformity_p:.2f}")

In [ ]:
fig, ax = plt.subplots(figsize=(5, 3))
ax.hist(p_values, bins=20, range=(0, 1), color="tab:orange", edgecolor="black")
ax.axhline(N_REALIZATIONS / 20, color="black", linestyle="--", label="expected if uniform")
ax.set(xlabel="KS p-value", ylabel="Number of realizations", title="Irregular neuron: KS p-values")
ax.legend()
fig.tight_layout()
plt.show()

**Why not join all ISIs into one big test?** It is tempting to pool the ~50,000 ISIs of all
realizations and run a single KS test. We tried it, and it **fails** even though the model is correct
(see the cell below). The reason is the 5 s recording window: a long interval is more likely to be cut
by the end of the window, so the ISIs we can measure are slightly shorter than the true ones
(mean about 98 ms instead of 100 ms). With 50,000 values the KS test detects this tiny bias.
This is a property of *measuring* inside a finite window, not an error in the model – testing each
realization separately avoids it.

In [ ]:
# the tempting (but misleading) pooled test
pooled_isis = np.concatenate([np.diff(t) for t in realizations["Irregular"]])
pooled_p = stats.kstest(pooled_isis, theory_isi.cdf).pvalue
print(f"Pooled test: {len(pooled_isis)} ISIs, mean = {pooled_isis.mean() * 1000:.1f} ms, KS p = {pooled_p:.1e}")

## [P1] Q1 C5. (main edge-case check) Changing the recording duration
**Question:** what happens when the recording is much shorter or much longer than 5 s?
(Assignment: *"What happens when the recording duration is changed?"*)

Every model function takes the recording duration `T` as an argument, so we simply pass a different `T`.

**5a. Extremely short recording (10 ms) – zero or one spike.**
Nothing guarantees a spike in such a short window, so the code must handle empty and one-spike trains
without crashing. For the regular neuron we can even predict the result: its first spike is uniform in
[0, 100 ms) and the next one comes ~100 ms later, so a 10 ms window contains **one spike with
probability 10 %** and **zero spikes otherwise** – never two.


**5b. Longer recordings (0.5 s → 500 s) – the rate estimate converges.**
The firing rate estimated from one recording (spikes / duration) should approach 10 Hz, and its
spread between realizations should shrink like 1/√T. For a renewal process, theory gives
SD(rate) ≈ √(RATE) · CV / √T.

In [ ]:
# ---- 5a: a 10 ms recording ----
T_SHORT = 0.010
spike_counts = {}
for name, simulate in SIMULATORS.items():
    rng = np.random.default_rng(CHECK_SEED)
    # if anything breaks on 0 or 1 spikes, this line raises an error
    spike_counts[name] = np.array([len(simulate(T_SHORT, rng, rate_hz=RATE)) for _ in range(N_REALIZATIONS)])

for name, counts in spike_counts.items():
    print(f"{name:10s}: 0 spikes in {np.mean(counts == 0) * 100:4.1f} %, "
          f"1 spike in {np.mean(counts == 1) * 100:4.1f} %, max = {counts.max()}")

low, high = stats.binom.ppf([0.025, 0.975], N_REALIZATIONS, 0.10) / N_REALIZATIONS
fraction_one = np.mean(spike_counts["Regular"] == 1)
report("5a very short recording (10 ms)",
       spike_counts["Regular"].max() <= 1 and low <= fraction_one <= high,
       f"no errors for 0/1-spike trains; Regular: 1 spike in {fraction_one * 100:.1f} % "
       f"(expected {low * 100:.1f}-{high * 100:.1f} %), max {spike_counts['Regular'].max()} spike")

In [ ]:
# ---- 5b: longer recordings, irregular neuron ----
DURATIONS = [0.5, 5, 50, 500]
cv_irregular_theory = (MEAN_ISI - DEAD_TIME) / MEAN_ISI
sd_theory_scaled = np.sqrt(RATE) * cv_irregular_theory   # SD(rate) * sqrt(T) predicted by theory

rate_estimates = {}
for T in DURATIONS:
    rng = np.random.default_rng(CHECK_SEED)
    rate_estimates[T] = np.array([len(simulate_irregular(T, rng, rate_hz=RATE)) / T for _ in range(N_REALIZATIONS)])

for T in DURATIONS:
    rates = rate_estimates[T]
    standard_error = rates.std(ddof=1) / np.sqrt(N_REALIZATIONS)
    sd_scaled = rates.std(ddof=1) * np.sqrt(T)
    details = (f"mean rate = {rates.mean():.2f} Hz, SD = {rates.std(ddof=1):.3f} Hz, "
               f"SD·√T = {sd_scaled:.2f} (theory {sd_theory_scaled:.2f})")
    if T == 0.5:
        print(f"[info] 5b T = {T} s: {details}")
        continue
    passed = (abs(rates.mean() - RATE) <= 3 * standard_error) and \
             (abs(sd_scaled / sd_theory_scaled - 1) <= 0.15)
    report(f"5b rate converges (T = {T} s)", passed, details)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 3.5))
sds = [rate_estimates[T].std(ddof=1) for T in DURATIONS]
T_line = np.logspace(np.log10(0.4), np.log10(600), 100)
ax.plot(T_line, sd_theory_scaled / np.sqrt(T_line), color="black", label="theory: ∝ 1/√T")
ax.plot(DURATIONS, sds, "o", color="tab:orange", label="irregular neuron, 1000 realizations")
ax.set(xscale="log", yscale="log", xlabel="Recording duration T (s)",
       ylabel="SD of estimated rate (Hz)", title="Rate estimate gets more precise with longer recordings")
ax.legend()
fig.tight_layout()
fig.savefig(FIG_DIR / "q1_check_duration.png", dpi=200)
plt.show()

## [P1] Q1 C6. Summary of all checks

In [ ]:
import pandas as pd
pd.DataFrame(results)

**Conclusion.** The three main checks – one temporal (2.2), one statistical (4.1) and one edge case
(5) – together with the supporting checks show that the simulation implements the intended models:
spike times are always valid and ordered, the regular neuron is regular, bursts never merge, the
variability is controlled by `k` exactly as the theory predicts, the irregular ISIs have the intended
exponential shape, the code handles recordings with zero or one spike, and results are reproducible.
Two checks were confirmed to be *able to fail*: the regularity test rejects the irregular neuron,
and the burst test detects a model without the 50 ms quiet period.

## [P1] Q1 interpretation – what distinguishes the three neurons
*(Decision `d_report`: at integration this chapter goes at the **end** of `Q1_neural_spikes.ipynb`, after Q1 D,
and the parameter table of section A11 goes to the **top** of the notebook.)*

The numbers below are computed from the 1000 realizations of 5 s used in Q1 C (same models, seed `CHECK_SEED`).
For each measure: the median and, in brackets, the range that contains 90 % of the realizations.

In [ ]:
def lag1_correlation(spike_times):
    """Correlation between each ISI and the next one."""
    isis = np.diff(spike_times)
    return np.corrcoef(isis[:-1], isis[1:])[0, 1]

MEASURES = {
    "rate (Hz)": lambda t: len(t) / DURATION,
    "CV of ISI": cv_of_train,
    "fraction of ISIs < 10 ms": lambda t: np.mean(np.diff(t) < 0.010),
    "lag-1 ISI correlation": lag1_correlation,
}

rows = {}
for name, trains in realizations.items():
    rows[name] = {}
    for measure, function in MEASURES.items():
        values = np.array([function(t) for t in trains])
        low, median, high = np.percentile(values, [5, 50, 95])
        rows[name][measure] = f"{median:.3g} [{low:.3g}, {high:.3g}]"

pd.DataFrame(rows).T

### 1. The models and their main assumptions
* All three neurons fire at the **same mean rate (10 Hz)** with a 2 ms refractory period, so every
  difference between them comes from the **timing** of the spikes (table in section A11).
* The regular and irregular neurons are **renewal processes**: each interval is drawn independently
  from a Gamma distribution, and only its shape `k` differs (400 vs 1). The bursting neuron has two
  kinds of intervals – about 5 ms inside a burst and at least 50 ms (on average ~385 ms) between bursts.
* The parameters were **chosen** to represent typical firing patterns, not fitted to recorded data.
  The models leave out properties of real neurons such as adaptation (slowing down over time),
  changes of rate during the recording, and dependence between successive intervals of the regular
  and irregular neurons.

### 2. The mean firing rate alone cannot tell them apart
In a 5 s recording the spike count of the irregular and bursting neurons varies a lot between
realizations (rate between about 7.5 and 12.5 Hz), and the ranges of the three neurons overlap.
The same rate is produced by evenly spaced, random, and clustered spikes (raster in section A12).

### 3. What does distinguish them: the distribution of the intervals
* **CV of the ISIs:** about 0.05 (regular), about 1 (irregular, as expected for a Poisson-like
  process) and about 2 (bursting). The 90 % ranges do not overlap.
* **Fraction of very short ISIs (< 10 ms):** 0 for the regular neuron, a few percent for the
  irregular one (short intervals happen by chance) and about 0.75 for the bursting one.
* **Shape of the ISI histogram:** one narrow peak at 100 ms (regular), highest at short intervals and
  decaying (irregular), two separate peaks on a log axis (bursting) – sections A8–A10.
* **Order of the intervals:** only the bursting neuron shows a clearly negative lag-1 correlation
  (about −0.26): a long quiet interval is usually followed by a short one inside the next burst.
  For the renewal neurons it is close to 0, as the model implies.

### 4. Variability between realizations
Two realizations of the irregular or bursting neuron can look quite different – different spike
counts and spikes or bursts in different places. But the **shape measures** (CV, fraction of short
ISIs) are stable: one 5 s recording is usually enough to place a neuron. Q1 D tests this on new
simulations with a simple threshold rule.

### 5. How we would tell the patterns apart from observations alone
Without knowing how the data were generated:
1. Compute the ISIs and plot their histogram on a **log axis**.
2. If a large fraction of the ISIs is very short (here around 0.75 below 10 ms) and the histogram has
   **two separate peaks**, the neuron is **bursting**.
3. Otherwise, if the **CV is far below 1** and the histogram is one narrow peak, it is **regular**.
4. Otherwise, if the **CV is close to 1** and the ISI histogram decays exponentially, it is
   **irregular** (Poisson-like). A KS test against an exponential distribution, done as in C4.2, can
   support this.

### 6. How we know the results are trustworthy, and what we cannot conclude
* The checks of Q1 C show that the code implements the intended models: valid and ordered spike
  times, a regular neuron that is really regular, bursts that never merge, variability that follows
  the theory for every `k`, the intended exponential shape for the irregular neuron, correct
  behaviour for recordings of 10 ms to 500 s, and reproducible results.
* One check (C5) **found a real problem** in our first version: starting every neuron at a random time
  in its first interval added about half a spike to each recording of the irregular neuron. A 2 s
  burn-in fixed it. Plausible-looking spike trains were not enough to detect this.
* Measurements from a short window are biased: in 5 s, long intervals are more likely to be cut, so
  the measured mean ISI is slightly too short (C4.2). Numbers from one short recording should be
  compared with their spread over realizations, not with the exact theoretical value.
* The thresholds work for **these three models with equal rates**. Real neurons can sit between the
  categories (for example a CV of 0.5, or bursts of only 2 spikes), so the thresholds would have to be
  re-checked on real data; we cannot conclude that they separate real neurons equally well.